# 📊 Dashboard Completo de Portfólio

Dashboard unificado com análise completa do portfólio e dividendos.

### Secções
- ⚙️ **Configuração** — portfólio e parâmetros
- 📦 **Resumo Geral** — valor investido, atual, lucro/prejuízo
- 📈 **Evolução Histórica** — valor do portfólio ao longo do tempo
- 🥧 **Composição** — distribuição por ativo e por setor
- 📊 **Performance** — retorno por ação e performance normalizada
- 🔥 **Correlação** — mapa de correlação entre ativos
- 💰 **Dividendos** — histórico, yield, calendário mensal
- 🔮 **Projeção** — renda futura com dividendos

---
**Dependências:**
```bash
pip install yfinance pandas plotly numpy
```

---
## 📦 Célula 1 — Importações

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas carregadas com sucesso!')

✅ Bibliotecas carregadas com sucesso!


---
## ⚙️ Célula 2 — Configuração Central

> **✏️ Edite apenas aqui!** Todos os gráficos e cálculos usam esta configuração.

In [2]:
# ============================================================
# 🔧 PORTFÓLIO — edite com seus dados reais
#    shares    : quantidade de cotas que possui
#    avg_price : preço médio de compra em dólares
#    sector    : setor do ativo
# ============================================================
PORTFOLIO = {
    # — Tecnologia —
    "AAPL":   {"shares": 1.013,   "avg_price": 198.03, "sector": "Tecnologia"},
    "ASML":   {"shares": 0.3281,  "avg_price": 640.02, "sector": "Tecnologia"},
    "GOOGL":  {"shares": 0.7847,  "avg_price": 273.98, "sector": "Tecnologia"},
    "EMBJ":   {"shares": 1,       "avg_price": 46.53,  "sector": "Tecnologia"},
    "AVGO":   {"shares": 0.3659,  "avg_price": 210.00, "sector": "Tecnologia"},
    # — Energia —
    "ENB":    {"shares": 4.9681,  "avg_price": 48.66,  "sector": "Energia"},
    "SO":     {"shares": 2.1269,  "avg_price": 94.82,  "sector": "Energia"},
    # — Saúde —
    "JNJ":    {"shares": 1,       "avg_price": 157.94, "sector": "Saúde"},
    "CVS":    {"shares": 3,       "avg_price": 74.38,  "sector": "Saúde"},
    # — Bem de Consumo —
    "KO":     {"shares": 2,       "avg_price": 72.38,  "sector": "Bem de Consumo"},
    "PG":     {"shares": 1.4796,  "avg_price": 150.07, "sector": "Bem de Consumo"},
    # — REITs —
    "NTST":   {"shares": 4,       "avg_price": 18.17,  "sector": "REITs"},
    "O":      {"shares": 5,       "avg_price": 57.57,  "sector": "REITs"},
    "EQIX":   {"shares": 0.2975,  "avg_price": 843.36, "sector": "REITs"},
    "PLD":    {"shares": 2,       "avg_price": 116.21, "sector": "REITs"},
    "WPC":    {"shares": 2,       "avg_price": 61.95,  "sector": "REITs"},
    # — Finanças —
    "NU":     {"shares": 16.7574, "avg_price": 14.69,  "sector": "Finanças"},
    "BDORY":  {"shares": 10,      "avg_price": 4.70,   "sector": "Finanças"},
    "JPM":    {"shares": 0.3685,  "avg_price": 312.00, "sector": "Finanças"},
    "V":      {"shares": 0.8597,  "avg_price": 325.65, "sector": "Finanças"},
    # — ETFs —
    "CSPX.L": {"shares": 0.2821,  "avg_price": 744.30, "sector": "ETF"},
    "VWRA.L": {"shares": 1.094,   "avg_price": 158.98, "sector": "ETF"},
    "XGDU.L": {"shares": 4,       "avg_price": 66.07,  "sector": "ETF"},
}

# ============================================================
# ⚙️ PARÂMETROS GERAIS
# ============================================================
PERIODO          = "1y"   # histórico de preços: '1mo','3mo','6mo','1y','2y','5y'
ANOS_DIVIDENDOS  = 3      # anos de histórico de dividendos
ANOS_PROJECAO    = 10     # anos de projeção de renda futura
CRESCIMENTO_DIV  = 0.05   # crescimento anual assumido dos dividendos (5%)

# ============================================================
TICKERS = list(PORTFOLIO.keys())
print(f'📋 {len(TICKERS)} ativos carregados: {", ".join(TICKERS)}')

📋 23 ativos carregados: AAPL, ASML, GOOGL, EMBJ, AVGO, ENB, SO, JNJ, CVS, KO, PG, NTST, O, EQIX, PLD, WPC, NU, BDORY, JPM, V, CSPX.L, VWRA.L, XGDU.L


---
## 🌐 Célula 3 — Download de Dados

Baixa preços atuais, históricos e histórico de dividendos via `yfinance`.

In [3]:
def get_portfolio_data(portfolio: dict) -> pd.DataFrame:
    """Preço atual, variação diária e lucro/prejuízo por ativo."""
    rows = []
    print('⏳ Buscando preços atuais...')
    for ticker, info in portfolio.items():
        try:
            hist = yf.Ticker(ticker).history(period='2d')
            if hist.empty:
                print(f'  ⚠️  Sem dados para {ticker}')
                continue
            preco_atual = hist['Close'].iloc[-1]
            preco_ant   = hist['Close'].iloc[-2] if len(hist) > 1 else preco_atual
            var_dia     = ((preco_atual - preco_ant) / preco_ant) * 100
            shares      = info['shares']
            avg         = info['avg_price']
            investido   = shares * avg
            atual       = shares * preco_atual
            lucro       = atual - investido
            lucro_pct   = ((atual / investido) - 1) * 100
            rows.append({
                'Ticker':              ticker,
                'Setor':               info['sector'],
                'Preço Atual ($)':     round(preco_atual, 2),
                'Preço Médio ($)':     round(avg, 2),
                'Cotas':               shares,
                'Valor Investido ($)': round(investido, 2),
                'Valor Atual ($)':     round(atual, 2),
                'Lucro/Prej. ($)':     round(lucro, 2),
                'Lucro/Prej. (%)':     round(lucro_pct, 2),
                'Var. Diária (%)':     round(var_dia, 2),
            })
            print(f'  ✅ {ticker}: ${preco_atual:.2f}  ({var_dia:+.2f}% hoje)')
        except Exception as e:
            print(f'  ❌ Erro em {ticker}: {e}')
    return pd.DataFrame(rows)


def get_historical_prices(tickers: list, period: str) -> pd.DataFrame:
    """Preços históricos ajustados para todos os tickers."""
    data = yf.download(tickers, period=period, auto_adjust=True, progress=False)['Close']
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])
    return data


def get_dividendos(portfolio: dict, anos: int) -> pd.DataFrame:
    """Histórico de dividendos recebidos por ativo."""
    data_inicio = datetime.now() - timedelta(days=365 * anos)
    rows = []
    sem_dividendo = []
    print('\n⏳ Buscando histórico de dividendos...')
    for ticker, info in portfolio.items():
        try:
            divs = yf.Ticker(ticker).dividends
            if divs.empty:
                sem_dividendo.append(ticker)
                continue
            divs.index = divs.index.tz_localize(None)
            divs = divs[divs.index >= data_inicio]
            if divs.empty:
                sem_dividendo.append(ticker)
                continue
            for data, valor in divs.items():
                rows.append({
                    'Ticker':         ticker,
                    'Setor':          info['sector'],
                    'Data':           data,
                    'Dividendo/Cota': round(valor, 4),
                    'Cotas':          info['shares'],
                    'Recebido ($)':   round(valor * info['shares'], 4),
                    'Ano':            data.year,
                    'Mês':            data.month,
                    'Mês/Ano':        data.strftime('%b/%Y'),
                })
            print(f'  ✅ {ticker}: {len(divs)} pagamentos')
        except Exception as e:
            print(f'  ❌ Erro em {ticker}: {e}')
    if sem_dividendo:
        print(f'  ⚪ Sem dividendos: {", ".join(sem_dividendo)}')
    return pd.DataFrame(rows)


def get_yield_info(portfolio: dict, df_divs: pd.DataFrame) -> pd.DataFrame:
    """Dividend yield e projeção anual por ativo."""
    rows = []
    ultimo_ano = datetime.now().year - 1
    for ticker, info in portfolio.items():
        try:
            stock       = yf.Ticker(ticker)
            preco       = stock.fast_info.last_price
            div_forward = stock.info.get('dividendRate', 0) or 0
            dy          = div_forward / preco if preco and preco > 0 and div_forward > 0 else 0
            recebido    = df_divs.loc[
                (df_divs['Ticker'] == ticker) & (df_divs['Ano'] == ultimo_ano),
                'Recebido ($)'
            ].sum()
            projecao    = div_forward * info['shares']
            rows.append({
                'Ticker':              ticker,
                'Setor':               info['sector'],
                'Cotas':               info['shares'],
                'Preço Atual ($)':     round(preco, 2),
                'Div. Forward/Cota':   round(div_forward, 4),
                'Yield Atual (%)':     round(dy * 100, 2),
                f'Recebido {ultimo_ano} ($)': round(recebido, 2),
                'Projeção Anual ($)':  round(projecao, 2),
                'Projeção Mensal ($)': round(projecao / 12, 2),
            })
        except Exception as e:
            print(f'Erro em {ticker}: {e}')
    return pd.DataFrame(rows).sort_values('Projeção Anual ($)', ascending=False)


# — Executa todos os downloads —
df        = get_portfolio_data(PORTFOLIO)
prices    = get_historical_prices(TICKERS, PERIODO)
df_divs   = get_dividendos(PORTFOLIO, ANOS_DIVIDENDOS)
df_yield  = get_yield_info(PORTFOLIO, df_divs)

print(f'\n📅 Histórico de {PERIODO}: {len(prices)} pregões')
print(f'💰 Dividendos registados: {len(df_divs)} pagamentos')

⏳ Buscando preços atuais...
  ✅ AAPL: $264.15  (-0.86% hoje)
  ✅ ASML: $1419.00  (-4.24% hoje)
  ✅ GOOGL: $335.08  (-0.61% hoje)
  ✅ EMBJ: $65.88  (-3.35% hoje)
  ✅ AVGO: $397.93  (+0.30% hoje)
  ✅ ENB: $52.43  (-0.34% hoje)
  ✅ SO: $94.58  (-0.06% hoje)
  ✅ JNJ: $233.74  (-2.07% hoje)
  ✅ CVS: $76.43  (+1.92% hoje)
  ✅ KO: $75.08  (-0.31% hoje)
  ✅ PG: $143.18  (-0.14% hoje)
  ✅ NTST: $20.45  (+1.34% hoje)
  ✅ O: $64.29  (+0.52% hoje)
  ✅ EQIX: $1069.24  (+1.54% hoje)
  ✅ PLD: $142.30  (+1.81% hoje)
  ✅ WPC: $72.42  (+1.17% hoje)
  ✅ NU: $15.40  (+0.39% hoje)
  ✅ BDORY: $4.88  (-0.81% hoje)
  ✅ JPM: $308.36  (+0.79% hoje)
  ✅ V: $316.15  (+0.08% hoje)
  ✅ CSPX.L: $756.16  (+0.63% hoje)
  ✅ VWRA.L: $178.60  (+0.39% hoje)
  ✅ XGDU.L: $73.89  (+0.11% hoje)

⏳ Buscando histórico de dividendos...
  ✅ AAPL: 12 pagamentos
  ✅ ASML: 12 pagamentos
  ✅ GOOGL: 8 pagamentos
  ✅ EMBJ: 1 pagamentos
  ✅ AVGO: 12 pagamentos
  ✅ ENB: 11 pagamentos
  ✅ SO: 12 pagamentos
  ✅ JNJ: 12 pagamentos
  ✅ CVS: 

---
## 📋 Célula 4 — Resumo Geral do Portfólio

In [4]:
total_investido = df['Valor Investido ($)'].sum()
total_atual     = df['Valor Atual ($)'].sum()
lucro_total     = total_atual - total_investido
lucro_pct       = ((total_atual / total_investido) - 1) * 100
sinal           = '+' if lucro_total >= 0 else ''

print('=' * 48)
print('         📊 RESUMO DO PORTFÓLIO')
print('=' * 48)
print(f'  💰 Total Investido  :  ${total_investido:>12,.2f}')
print(f'  📈 Valor Atual      :  ${total_atual:>12,.2f}')
print(f'  💹 Lucro/Prejuízo   :  {sinal}${lucro_total:>11,.2f}  ({sinal}{lucro_pct:.2f}%)')
print(f'  🎯 Nº de Ativos     :  {len(df)}')
print('=' * 48)

df.style \
    .map(lambda v: 'color: green' if isinstance(v, float) and v > 0
                   else 'color: red' if isinstance(v, float) and v < 0 else '',
         subset=['Lucro/Prej. ($)', 'Lucro/Prej. (%)', 'Var. Diária (%)']) \
    .format({
        'Preço Atual ($)':     '${:.2f}',
        'Preço Médio ($)':     '${:.2f}',
        'Valor Investido ($)': '${:,.2f}',
        'Valor Atual ($)':     '${:,.2f}',
        'Lucro/Prej. ($)':     '${:+,.2f}',
        'Lucro/Prej. (%)':     '{:+.2f}%',
        'Var. Diária (%)':     '{:+.2f}%',
    })

         📊 RESUMO DO PORTFÓLIO
  💰 Total Investido  :  $    4,244.27
  📈 Valor Atual      :  $    5,041.62
  💹 Lucro/Prejuízo   :  +$     797.35  (+18.79%)
  🎯 Nº de Ativos     :  23


,Ticker,Setor,Preço Atual ($),Preço Médio ($),Cotas,Valor Investido ($),Valor Atual ($),Lucro/Prej. ($),Lucro/Prej. (%),Var. Diária (%)
0,AAPL,Tecnologia,$264.15,$198.03,1.013000,$200.60,$267.58,$+66.98,+33.39%,-0.86%
1,ASML,Tecnologia,$1419.00,$640.02,0.328100,$209.99,$465.57,$+255.58,+121.71%,-4.24%
2,GOOGL,Tecnologia,$335.08,$273.98,0.784700,$214.99,$262.94,$+47.95,+22.30%,-0.61%
3,EMBJ,Tecnologia,$65.88,$46.53,1.000000,$46.53,$65.88,$+19.35,+41.59%,-3.35%
4,AVGO,Tecnologia,$397.93,$210.00,0.365900,$76.84,$145.60,$+68.76,+89.49%,+0.30%
5,ENB,Energia,$52.43,$48.66,4.968100,$241.75,$260.48,$+18.73,+7.75%,-0.34%
6,SO,Energia,$94.58,$94.82,2.126900,$201.67,$201.16,$-0.51,-0.25%,-0.06%
7,JNJ,Saúde,$233.74,$157.94,1.000000,$157.94,$233.74,$+75.80,+47.99%,-2.07%
8,CVS,Saúde,$76.43,$74.38,3.000000,$223.14,$229.29,$+6.15,+2.76%,+1.92%
9,KO,Bem de Consumo,$75.08,$72.38,2.000000,$144.76,$150.16,$+5.40,+3.73%,-0.31%


---
## 📈 Célula 5 — Evolução Histórica do Portfólio

In [5]:
portfolio_value = pd.Series(0.0, index=prices.index)
for ticker, info in PORTFOLIO.items():
    if ticker in prices.columns:
        portfolio_value += prices[ticker] * info['shares']

custo_base = sum(v['shares'] * v['avg_price'] for v in PORTFOLIO.values())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=portfolio_value.index,
    y=portfolio_value.values,
    mode='lines',
    name='Valor do Portfólio',
    line=dict(color='#00d4aa', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(0,212,170,0.12)',
))
fig.add_hline(
    y=custo_base,
    line_dash='dash',
    line_color='#ff4d6d',
    annotation_text=f' Custo Base: ${custo_base:,.0f}',
    annotation_position='bottom right',
)
fig.update_layout(
    title=f'Evolução do Portfólio — últimos {PERIODO}',
    xaxis_title='Data', yaxis_title='Valor ($)',
    yaxis_tickprefix='$', template='plotly_dark',
    hovermode='x unified', height=450,
)
fig.show()

---
## 🥧 Célula 6 — Composição do Portfólio

In [6]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Por Ativo', 'Por Setor'],
)

# Pizza por ativo
fig.add_trace(go.Pie(
    labels=df['Ticker'],
    values=df['Valor Atual ($)'],
    hole=0.45,
    textposition='outside',
    textinfo='percent+label',
    marker_colors=px.colors.sequential.Teal,
), row=1, col=1)

# Pizza por setor
por_setor = df.groupby('Setor')['Valor Atual ($)'].sum().reset_index()
fig.add_trace(go.Pie(
    labels=por_setor['Setor'],
    values=por_setor['Valor Atual ($)'],
    hole=0.45,
    textposition='outside',
    textinfo='percent+label',
    marker_colors=px.colors.sequential.Plasma_r,
), row=1, col=2)

fig.update_layout(
    title='Composição do Portfólio (valor atual)',
    template='plotly_dark', height=500,
    showlegend=False,
)
fig.show()

---
## 📊 Célula 7 — Performance por Ação

In [7]:
df_sorted = df.sort_values('Lucro/Prej. (%)')
cores = ['#00d4aa' if v >= 0 else '#ff4d6d' for v in df_sorted['Lucro/Prej. (%)']]

fig = go.Figure(go.Bar(
    x=df_sorted['Ticker'],
    y=df_sorted['Lucro/Prej. (%)'],
    marker_color=cores,
    text=[f'{v:+.2f}%' for v in df_sorted['Lucro/Prej. (%)']],
    textposition='outside',
))
fig.update_layout(
    title='Performance Total por Ação',
    xaxis_title='Ticker', yaxis_title='Retorno (%)',
    yaxis_ticksuffix='%', template='plotly_dark', height=420,
)
fig.show()

---
## 📉 Célula 8 — Performance Normalizada (base 100)

In [8]:
prices_norm = (prices / prices.iloc[0]) * 100

fig = go.Figure()
for ticker in TICKERS:
    if ticker in prices_norm.columns:
        fig.add_trace(go.Scatter(
            x=prices_norm.index,
            y=prices_norm[ticker],
            mode='lines', name=ticker,
        ))
fig.add_hline(y=100, line_dash='dot', line_color='gray', annotation_text=' Base 100')
fig.update_layout(
    title=f'Performance Normalizada (base 100) — {PERIODO}',
    xaxis_title='Data', yaxis_title='Valor Normalizado',
    template='plotly_dark', hovermode='x unified', height=450,
)
fig.show()

---
## 🔥 Célula 9 — Mapa de Correlação

> **+1** = movem juntos &nbsp;|&nbsp; **−1** = movem ao contrário &nbsp;|&nbsp; Correlações baixas = melhor diversificação

In [9]:
returns = prices.pct_change().dropna()
corr    = returns.corr()

fig = px.imshow(
    corr, text_auto='.2f',
    title=f'Correlação dos Retornos Diários — {PERIODO}',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, aspect='auto',
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

mascara   = np.triu(np.ones(corr.shape), k=1).astype(bool)
stacked   = corr.where(mascara).stack()
stacked.index.names = ['Ativo A', 'Ativo B']
corr_pairs = stacked.reset_index(name='Correlação')
menor = corr_pairs.loc[corr_pairs['Correlação'].idxmin()]
maior = corr_pairs.loc[corr_pairs['Correlação'].idxmax()]
print(f'🎯 Menor correlação : {menor["Ativo A"]} × {menor["Ativo B"]} = {menor["Correlação"]:.2f}')
print(f'🔗 Maior correlação : {maior["Ativo A"]} × {maior["Ativo B"]} = {maior["Correlação"]:.2f}')

🎯 Menor correlação : AVGO × PG = -0.30
🔗 Maior correlação : CSPX.L × VWRA.L = 0.95


---
## 💰 Célula 10 — Resumo de Dividendos por Ativo

In [10]:
total_projecao_anual  = df_yield['Projeção Anual ($)'].sum()
total_projecao_mensal = total_projecao_anual / 12

print('=' * 50)
print('       💰 RESUMO DE DIVIDENDOS')
print('=' * 50)
print(f'  📅 Projeção Anual   : ${total_projecao_anual:.2f}')
print(f'  📆 Projeção Mensal  : ${total_projecao_mensal:.2f}')
print('=' * 50)

df_yield.style \
    .map(lambda v: 'color: #00d4aa' if isinstance(v, (int, float)) and v > 0 else '',
         subset=['Projeção Anual ($)', 'Projeção Mensal ($)', 'Yield Atual (%)']) \
    .format({
        'Preço Atual ($)':     '${:.2f}',
        'Div. Forward/Cota':   '${:.4f}',
        'Yield Atual (%)':     '{:.2f}%',
        'Projeção Anual ($)':  '${:.2f}',
        'Projeção Mensal ($)': '${:.2f}',
    })

       💰 RESUMO DE DIVIDENDOS
  📅 Projeção Anual   : $96.75
  📆 Projeção Mensal  : $8.06


,Ticker,Setor,Cotas,Preço Atual ($),Div. Forward/Cota,Yield Atual (%),Recebido 2025 ($),Projeção Anual ($),Projeção Mensal ($)
12,O,REITs,5.000000,$64.29,$3.2400,5.04%,17.450000,$16.20,$1.35
5,ENB,Energia,4.968100,$52.42,$2.8400,5.42%,10.100000,$14.11,$1.18
14,PLD,REITs,2.000000,$142.34,$4.2800,3.01%,8.080000,$8.56,$0.71
8,CVS,Saúde,3.000000,$76.43,$2.6600,3.48%,7.980000,$7.98,$0.67
15,WPC,REITs,2.000000,$72.43,$3.7200,5.14%,7.240000,$7.44,$0.62
6,SO,Energia,2.126900,$94.59,$2.9600,3.13%,6.250000,$6.30,$0.52
10,PG,Bem de Consumo,1.479600,$143.19,$4.2600,2.98%,6.180000,$6.30,$0.53
13,EQIX,REITs,0.297500,$1069.24,$19.2300,1.80%,5.580000,$5.72,$0.48
7,JNJ,Saúde,1.000000,$233.73,$5.3600,2.29%,5.140000,$5.36,$0.45
9,KO,Bem de Consumo,2.000000,$75.08,$2.0600,2.74%,4.080000,$4.12,$0.34


---
## 📅 Célula 11 — Calendário Mensal de Dividendos

In [11]:
mensal = (
    df_divs.groupby(['Ano', 'Mês', 'Mês/Ano'])['Recebido ($)']
    .sum().reset_index().sort_values(['Ano', 'Mês'])
)

fig = go.Figure(go.Bar(
    x=mensal['Mês/Ano'],
    y=mensal['Recebido ($)'],
    marker_color='#00d4aa',
    text=[f'${v:.2f}' for v in mensal['Recebido ($)']],
    textposition='outside',
))
fig.update_layout(
    title='💵 Dividendos Recebidos por Mês',
    xaxis_title='Mês', yaxis_title='Total Recebido ($)',
    yaxis_tickprefix='$', template='plotly_dark', height=420,
)
fig.show()

---
## 🥧 Célula 12 — Distribuição de Dividendos

In [12]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Por Ativo', 'Por Setor'],
)

por_ativo_div = df_divs.groupby('Ticker')['Recebido ($)'].sum().reset_index()
fig.add_trace(go.Pie(
    labels=por_ativo_div['Ticker'],
    values=por_ativo_div['Recebido ($)'],
    hole=0.45, textposition='outside',
    textinfo='percent+label',
    marker_colors=px.colors.sequential.Teal,
), row=1, col=1)

por_setor_div = df_divs.groupby('Setor')['Recebido ($)'].sum().reset_index()
fig.add_trace(go.Pie(
    labels=por_setor_div['Setor'],
    values=por_setor_div['Recebido ($)'],
    hole=0.45, textposition='outside',
    textinfo='percent+label',
    marker_colors=px.colors.sequential.Plasma_r,
), row=1, col=2)

fig.update_layout(
    title=f'Distribuição de Dividendos — últimos {ANOS_DIVIDENDOS} anos',
    template='plotly_dark', height=500, showlegend=False,
)
fig.show()

---
## 📊 Célula 13 — Dividend Yield por Ativo

In [13]:
df_yield_plot = df_yield[df_yield['Yield Atual (%)'] > 0].sort_values('Yield Atual (%)')

fig = go.Figure(go.Bar(
    x=df_yield_plot['Yield Atual (%)'],
    y=df_yield_plot['Ticker'],
    orientation='h',
    marker_color='#64ffda',
    text=[f'{v:.2f}%' for v in df_yield_plot['Yield Atual (%)']],
    textposition='outside',
))
fig.update_layout(
    title='📈 Dividend Yield Atual por Ativo',
    xaxis_title='Yield (%)', xaxis_ticksuffix='%',
    template='plotly_dark', height=520,
)
fig.show()

---
## 🔮 Célula 14 — Projeção de Renda Futura

In [14]:
ano_atual = datetime.now().year
anos      = list(range(ano_atual, ano_atual + ANOS_PROJECAO + 1))
renda_flat = [total_projecao_anual] * len(anos)
renda_cresc = [total_projecao_anual * (1 + CRESCIMENTO_DIV) ** i for i in range(len(anos))]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=anos, y=renda_flat,
    name='Sem crescimento',
    marker_color='#8892b0',
))
fig.add_trace(go.Scatter(
    x=anos, y=renda_cresc,
    name=f'+{CRESCIMENTO_DIV*100:.0f}% a.a. (reinvestimento)',
    mode='lines+markers',
    line=dict(color='#00d4aa', width=2.5),
    marker=dict(size=7),
))
fig.update_layout(
    title=f'🔮 Projeção de Renda com Dividendos — {ANOS_PROJECAO} anos',
    xaxis_title='Ano', yaxis_title='Dividendos Anuais ($)',
    yaxis_tickprefix='$', template='plotly_dark',
    height=420, legend=dict(x=0.02, y=0.98),
)
fig.show()

print(f'\n📌 Renda atual projetada : ${total_projecao_anual:.2f}/ano  |  ${total_projecao_mensal:.2f}/mês')
print(f'📌 Em {ANOS_PROJECAO} anos (+{CRESCIMENTO_DIV*100:.0f}%/a.a.) : ${renda_cresc[-1]:.2f}/ano  |  ${renda_cresc[-1]/12:.2f}/mês')


📌 Renda atual projetada : $96.75/ano  |  $8.06/mês
📌 Em 10 anos (+5%/a.a.) : $157.60/ano  |  $13.13/mês


---
## 📆 Célula 15 — Histórico de Dividendos por Ativo

In [15]:
fig = go.Figure()
for ticker in df_divs['Ticker'].unique():
    sub = df_divs[df_divs['Ticker'] == ticker].sort_values('Data')
    fig.add_trace(go.Scatter(
        x=sub['Data'], y=sub['Dividendo/Cota'],
        mode='lines+markers', name=ticker,
        marker=dict(size=5),
    ))
fig.update_layout(
    title='Histórico do Dividendo por Cota',
    xaxis_title='Data', yaxis_title='Dividendo/Cota ($)',
    yaxis_tickprefix='$', template='plotly_dark',
    hovermode='x unified', height=480,
)
fig.show()

---
## 💾 Célula 16 — Exportar para CSV

In [16]:
# Descomente as linhas abaixo para exportar

#timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Snapshot do portfólio
#df.to_csv(f'portfolio_{timestamp}.csv', index=False, sep=';', decimal=',')
#print(f'✅ Portfolio salvo: portfolio_{timestamp}.csv')

# Histórico de dividendos
#df_divs.to_csv(f'dividendos_historico_{timestamp}.csv', index=False, sep=';', decimal=',')
#print(f'✅ Dividendos salvos: dividendos_historico_{timestamp}.csv')

# Resumo de yield
#df_yield.to_csv(f'dividendos_resumo_{timestamp}.csv', index=False, sep=';', decimal=',')
#print(f'✅ Resumo yield salvo: dividendos_resumo_{timestamp}.csv')